In [ ]:
# --- Xray Cloud: Count Test Runs in a Test Plan + Daily Snapshot & Chart ---
import requests, getpass, json, math, collections, csv, os
from datetime import datetime
# plotting deps (only used at the end)
try:
    import matplotlib
    matplotlib.use("Agg")  # safe for headless / CI
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception as e:
    HAS_MPL = False
    print(f"[i] Plotting disabled (matplotlib not available: {e})")

PLAN_KEY = "PROJ-1003"   # <-- set to your plan key (e.g., "PROJ-1003")

XRAY_AUTH_URL    = "https://xray.cloud.getxray.app/api/v2/authenticate"
XRAY_GRAPHQL_URL = "https://xray.cloud.getxray.app/api/v2/graphql"

# TEMP: bypass SSL certificate verification to unblock now
VERIFY_SSL = False
if not VERIFY_SSL:
    from urllib3.exceptions import InsecureRequestWarning
    import urllib3; urllib3.disable_warnings(InsecureRequestWarning)

# ---- prompt for proper Xray API keys (NOT Atlassian PAT) ----
CID  = getpass.getpass("XRAY_CLIENT_ID (Xray → API Keys): ")
CSEC = getpass.getpass("XRAY_CLIENT_SECRET (Xray → API Keys): ")

def auth():
    r = requests.post(XRAY_AUTH_URL, json={"client_id": CID, "client_secret": CSEC},
                      timeout=30, verify=VERIFY_SSL)
    if r.status_code != 200:
        raise SystemExit(f"Auth failed: {r.status_code} {r.text[:300]}")
    return r.text.strip('"')

TOKEN   = auth()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

def gql(query, variables=None):
    r = requests.post(XRAY_GRAPHQL_URL, json={"query": query, "variables": variables or {}},
                      headers=HEADERS, timeout=60, verify=VERIFY_SSL)
    if r.status_code != 200:
        raise SystemExit(f"GraphQL HTTP {r.status_code}: {r.text[:300]}")
    data = r.json()
    if "errors" in data:
        raise SystemExit(f"GraphQL errors: {data['errors']}")
    return data["data"]

# 1) Resolve Test Plan KEY -> issueId
Q_PLANS = """
query($jql:String!, $limit:Int!, $start:Int!) {
  getTestPlans(jql:$jql, limit:$limit, start:$start) {
    total start limit
    results { issueId jira(fields:["key","summary"]) }
  }
}
"""
plans = gql(Q_PLANS, {"jql": f'key = "{PLAN_KEY}"', "limit": 1, "start": 0})
if not plans["getTestPlans"]["results"]:
    raise SystemExit(f"No Test Plan found for key {PLAN_KEY}")
PLAN_ID = plans["getTestPlans"]["results"][0]["issueId"]
PLAN_NAME = f'{plans["getTestPlans"]["results"][0]["jira"]["key"]} - {plans["getTestPlans"]["results"][0]["jira"]["summary"]}'

# 2) Page through Test Executions in the plan to collect their issueIds
Q_PLAN_EXECS = """
query($planId:String!, $limit:Int!, $start:Int!) {
  getTestPlan(issueId:$planId) {
    testExecutions(limit:$limit, start:$start) {
      total start limit
      results { issueId }
    }
  }
}
"""
exec_ids = []
start = 0
page = 100
while True:
    d = gql(Q_PLAN_EXECS, {"planId": PLAN_ID, "limit": page, "start": start})
    te = d["getTestPlan"]["testExecutions"]
    exec_ids += [x["issueId"] for x in te["results"]]
    start += te["limit"]
    if start >= te["total"]:
        break

# 3) Pull Test Runs by batches of up to 100 testExecIssueIds and count statuses
Q_RUNS = """
query($execIds:[String], $limit:Int!, $start:Int!) {
  getTestRuns(testExecIssueIds:$execIds, limit:$limit, start:$start) {
    total start limit
    results { status { name } }
  }
}
"""

counts = collections.Counter()
total_runs = 0

# getTestRuns itself pages over runs; also batch exec ids to avoid huge queries
BATCH = 50  # conservative batch of exec ids
for i in range(0, len(exec_ids) or 1, max(1, BATCH)):  # handle 0 execs gracefully
    batch = exec_ids[i:i+BATCH]
    start = 0
    limit = 100
    while True:
        data = gql(Q_RUNS, {"execIds": batch, "limit": limit, "start": start})
        runs = data["getTestRuns"]
        for tr in runs["results"]:
            sname = (tr.get("status") or {}).get("name") or "UNKNOWN"
            counts[sname] += 1
            total_runs += 1
        start += runs["limit"]
        if start >= runs["total"]:
            break

# Map to your canonical buckets (tweak labels if your instance customized them)
PASSED    = sum(counts.get(k, 0) for k in ("PASSED", "Pass", "OK"))
FAILED    = sum(counts.get(k, 0) for k in ("FAILED", "Fail"))
DESCOPED  = sum(counts.get(k, 0) for k in ("DESCOPED", "Not Executed (Descoped)", "SKIPPED (Descoped)"))
TODO      = sum(counts.get(k, 0) for k in ("TODO", "TO DO", "NOT_RUN", "Untested", "To Do"))

# Executed = tests that actually ran (passed/failed). Adjust if you consider others "executed".
executed = PASSED + FAILED
pass_pct = round((PASSED / executed * 100.0), 2) if executed else 0.0
completion=round((executed / total_runs * 100.0), 2) if executed else 0.0

summary = {
    "Test Plan": PLAN_NAME,
    "Totals (Test Runs)": {
        "Total": total_runs,
        "Passed": PASSED,
        "Failed": FAILED,
        "Descoped": DESCOPED,
        "TO DO": TODO,
        "Pass % (of executed)": pass_pct,
        "RT Completion %": completion,
    },
    "Other Statuses": {k: v for k, v in counts.items()
                       if k not in {"PASSED","Pass","OK","FAILED","Fail","DESCOPED","Not Executed (Descoped)","SKIPPED (Descoped)","TODO","To Do","TO DO","NOT_RUN","Untested"}}
}

print(json.dumps(summary, indent=2))

# Nice Markdown table to paste to Jira/Confluence/Slack
print("\n".join([
    f"### Test Plan Rollup — {PLAN_NAME}",
    "",
    "| Metric | Count |",
    "|---|---|",
    f"| Total | {summary['Totals (Test Runs)']['Total']} |",
    f"| Passed | {summary['Totals (Test Runs)']['Passed']} |",
    f"| Failed | {summary['Totals (Test Runs)']['Failed']} |",
    f"| Descoped | {summary['Totals (Test Runs)']['Descoped']} |",
    f"| To Do | {summary['Totals (Test Runs)']['TO DO']} |",
    f"| Pass % (of executed) | {summary['Totals (Test Runs)']['Pass % (of executed)']} |",
     f"| RT Completion % | {summary['Totals (Test Runs)']['RT Completion %']} |",
]))

# ----------------------------
# 4) Append a daily snapshot
# ----------------------------
def save_daily_snapshot(plan_key, totals):
    today = datetime.now().strftime("%Y-%m-%d")
    fname = f"xray_progress_{plan_key}.csv"
    row = {
        "date": today,
        "total": totals["Total"],
        "passed": totals["Passed"],
        "failed": totals["Failed"],
        "descoped": totals["Descoped"],
        "todo": totals["TO DO"],
        "Pass %": totals["Pass % (of executed)"],
        "RT Completion %": totals["Completion %"],
    }
    file_exists = os.path.exists(fname)
    with open(fname, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            w.writeheader()
        w.writerow(row)
    return fname

csv_path = save_daily_snapshot(PLAN_KEY, summary["Totals (Test Runs)"])
print(f"\n[+] Snapshot appended to {csv_path}")

# ----------------------------
# 5) Render the daily chart PNG
#    Colors & logic as requested:
#    - "Complete" = Pass (green) + Fail (red) stacked
#    - To-Do = yellow
#    - Descoped = grey
#    - Show thin Total line as target
# ----------------------------
def plot_progress(csv_file, plan_key, plan_name):
    import pandas as pd
    import numpy as np

    df = pd.read_csv(csv_file, parse_dates=["date"])
    df = df.sort_values("date")
    if df.empty:
        print("[!] No data to plot yet.")
        return None

    # Compute derived fields
    df["completed"] = df["passed"] + df["failed"] + df["descoped"]
    pass_rate = 0.0
    if df["total"].iloc[-1] > 0:
        pass_rate = (df["passed"].iloc[-1] / df["total"].iloc[-1]) * 100

    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(df))
    bar_width = 0.6

    # Requested colors
    color_pass = "green"
    color_fail = "red"
    color_todo = "yellow"
    color_descoped = "grey"

    ax.bar(x, df["passed"], width=bar_width, label="Passed (Complete)", color=color_pass)
    ax.bar(x, df["failed"], bottom=df["passed"], width=bar_width, label="Failed (Complete)", color=color_fail)
    ax.bar(x, df["todo"], bottom=df["passed"]+df["failed"], width=bar_width, label="To-Do", color=color_todo)
    ax.bar(x, df["descoped"], bottom=df["passed"]+df["failed"]+df["todo"], width=bar_width, label="Descoped", color=color_descoped)

    # Thin target line for total
    ax.plot(x, df["total"], marker="x", linestyle="-", linewidth=1, label="Total (Target)")

    ax.set_xticks(x)
    ax.set_xticklabels([d.strftime("%b %d") for d in df["date"]])
    ax.set_xlabel("Date")
    ax.set_ylabel("Test Runs (daily snapshot)")
    ax.set_title(f"Daily Test Execution Progress — {plan_name}  |  Pass rate: {pass_rate:.1f}%")
    ax.legend(loc="upper left", ncol=3, fontsize=8)
    fig.tight_layout()

    out_png = f"xray_progress_{plan_key}.png"
    fig.savefig(out_png, dpi=150)
    plt.close(fig)
    return out_png

png_path = None
if HAS_MPL:
    png_path = plot_progress(csv_path, PLAN_KEY, PLAN_NAME)
    if png_path:
        print(f"[+] Saved chart: {png_path}")
else:
    print("[i] Skipping chart render; CSV snapshot is still updated.")
